In [2]:
!pip install transformers datasets peft accelerate bitsandbytes trl -q


In [3]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU name: Tesla T4
GPU memory: 15.6 GB


In [6]:
import pandas as pd
from datasets import Dataset

# Create our own movie recommendation dataset
data = {
    "input": [
        "I want a mind-bending sci-fi movie",
        "Suggest a feel-good romantic movie",
        "I love Christopher Nolan films",
        "I want a scary horror movie",
        "Suggest something like The Dark Knight",
        "I want an animated movie for adults",
        "I love movies about space exploration",
        "Suggest a classic Bollywood movie",
        "I want a thriller with plot twists",
        "I love movies about artificial intelligence",
        "Suggest a comedy movie",
        "I want something emotional and deep",
        "I love action movies with great story",
        "Suggest a movie like Interstellar",
        "I want a book about self improvement",
        "Suggest a science fiction book",
        "I love mystery thriller books",
        "Suggest a book like Harry Potter",
        "I want a book about AI and future",
        "Suggest a motivational book for students",
    ],
    "output": [
        "Watch Inception (2010) — a heist inside dreams with stunning visuals and a mind-blowing plot.",
        "Try The Notebook (2004) or Crazy, Stupid, Love (2011) — both are heartwarming and beautifully made.",
        "You'll love Memento (2000) and The Prestige (2006) — both showcase his genius for non-linear storytelling.",
        "Watch Hereditary (2018) or The Conjuring (2013) — deeply unsettling and brilliantly crafted horror.",
        "Try V for Vendetta (2005) and Heat (1995) — both have complex characters and gripping narratives.",
        "Watch Spirited Away (2001) or Persepolis (2007) — both are profound animated masterpieces.",
        "Watch The Martian (2015) and Apollo 13 (1995) — both are gripping real-feeling space survival stories.",
        "Watch Dil Chahta Hai (2001) or Lagaan (2001) — timeless classics that defined an era.",
        "Try Gone Girl (2014) or Shutter Island (2010) — both will leave you questioning everything.",
        "Watch Ex Machina (2014) and Her (2013) — both explore AI consciousness in deeply human ways.",
        "Try The Grand Budapest Hotel (2014) or Superbad (2007) — witty, fun and highly rewatchable.",
        "Watch Schindler's List (1993) or The Green Mile (1999) — both are emotionally devastating masterpieces.",
        "Try Mad Max: Fury Road (2015) or John Wick (2014) — relentless action with surprising depth.",
        "Watch Arrival (2016) and Contact (1997) — both combine deep space with profound human emotion.",
        "Read Atomic Habits by James Clear — practical strategies to build good habits and break bad ones.",
        "Read Dune by Frank Herbert — an epic sci-fi universe about politics, religion and survival.",
        "Read Gone Girl by Gillian Flynn or The Girl with the Dragon Tattoo — both are unputdownable.",
        "Try the Percy Jackson series or The Name of the Wind by Patrick Rothfuss — magical and immersive.",
        "Read Life 3.0 by Max Tegmark — explores what AI means for the future of humanity.",
        "Read The Alchemist by Paulo Coelho or Ikigai — both are short, powerful reads for students.",
    ]
}

df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)
print(dataset)
print("\nFirst example:")
print(dataset[0])

Dataset({
    features: ['input', 'output'],
    num_rows: 20
})

First example:
{'input': 'I want a mind-bending sci-fi movie', 'output': 'Watch Inception (2010) — a heist inside dreams with stunning visuals and a mind-blowing plot.'}


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "facebook/opt-125m"  # Small 125M parameter model — perfect for free GPU

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model loaded! Parameters: {model.num_parameters():,}")
print(f"Device: {next(model.parameters()).device}")

Loading tokenizer...


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded! Parameters: 125,239,296
Device: cuda:0


In [8]:
# Format data into instruction format
def format_example(example):
    return {
        "text": f"### Instruction: {example['input']}\n### Response: {example['output']}"
    }

formatted_dataset = dataset.map(format_example)
print("Formatted dataset:")
print(formatted_dataset[0]['text'])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Formatted dataset:
### Instruction: I want a mind-bending sci-fi movie
### Response: Watch Inception (2010) — a heist inside dreams with stunning visuals and a mind-blowing plot.


In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenize
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

tokenized_dataset = formatted_dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["input", "output", "text"])
tokenized_dataset.set_format("torch")

print("Tokenized! Ready to train.")
print(f"Dataset size: {len(tokenized_dataset)}")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenized! Ready to train.
Dataset size: 20


In [11]:
# Training configuration - fixed
training_args = TrainingArguments(
    output_dir="./recobot",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-5,
    fp16=False,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training on GPU...")
trainer.train()
print("Training complete! 🎉")

Starting training on GPU...


Step,Training Loss
10,129.202917
20,368.663013
30,423.770361
40,393.427319
50,372.500952


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete! 🎉


In [12]:
# Test the fine-tuned model
def get_recommendation(question):
    prompt = f"### Instruction: {question}\n### Response:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = response.split("### Response:")[-1].strip()
    return response

# Test it!
print("Testing RecoBot...\n")
print("Q: I want a mind-bending sci-fi movie")
print("A:", get_recommendation("I want a mind-bending sci-fi movie"))
print()
print("Q: Suggest a book about self improvement")
print("A:", get_recommendation("Suggest a book about self improvement"))

Testing RecoBot...

Q: I want a mind-bending sci-fi movie
A: love,,,,,, movie,,,,,,,, movie,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,, movie,,,,,,,,,, movie,,,,,, movie,,,,,,,,, movie,,,,,,,,,, movie,, movie,,,,,, movie,

Q: Suggest a book about self improvement
A: love movie, movie,,,,,,,,,,,, movie,,,,,,,,,,,,, movie,,,,, movie,,,,,,,,,, movie, movie,, movie,,,,,, movie,,,,,, movie,,,,,,,,,,,,,,,,,,,,,,,,,, movie movie movie,,,,


In [13]:
def get_recommendation(question):
    prompt = f"### Instruction: {question}\n### Response:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.5,  # Prevents repeating words
            no_repeat_ngram_size=3,   # No repeating 3-word phrases
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("### Response:")[-1].strip()
    return response

print("Q: I want a mind-bending sci-fi movie")
print("A:", get_recommendation("I want a mind-bending sci-fi movie"))
print()
print("Q: Suggest a book about self improvement")
print("A:", get_recommendation("Suggest a book about self improvement"))

Q: I want a mind-bending sci-fi movie
A: love,able ofpiecesig human Dragondown master short initedoo Flynn Pauloai Coelian deep un Ik mysteryolis readsput powerful Away Gill grippingho future Gonep Spir ways consciousness Ex books something Tatt Perse explore intelligence deeply artificial2007 Machina2013 adults2015 an Her Alchemist about motivational thriller1995 like Max book space profound plot survival for Road John relentless great surprising Mad depth have Wick complex Fury story narratives to Wind bad series2010 Jacksonetta Patrick Knight horrorussf self magical build Potter characters or Harry Percy

Q: Suggest a book about self improvement
A: love, movie ofablepiecesig Dragon humandown short Ikited masterian in deep Flynn futureoo Co Paulo unelai reads mysteryputpolis powerful Away gripping Gill ways Perse Spir consciousnessho deeply books Gone Ex Tatt explore artificial intelligence something an Machina20072013 adults2015 Her Alchemist motivational thriller1995 like Max sci p

In [ ]:
import os
from groq import Groq

# Add your Groq API key
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def get_recommendation(question):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are RecoBot, an expert at recommending movies and books. Give specific, enthusiastic recommendations with brief reasons why. Keep responses under 80 words."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Test!
print("Q: I want a mind-bending sci-fi movie")
print("A:", get_recommendation("I want a mind-bending sci-fi movie"))
print()
print("Q: Suggest a book about self improvement")
print("A:", get_recommendation("Suggest a book about self improvement"))

In [2]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00


In [5]:
import os
from groq import Groq

# Add your Groq API key
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def get_recommendation(question):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are RecoBot, an expert at recommending movies and books. Give specific, enthusiastic recommendations with brief reasons why. Keep responses under 80 words."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Test!
print("Q: I want a mind-bending sci-fi movie")
print("A:", get_recommendation("I want a mind-bending sci-fi movie"))
print()
print("Q: Suggest a book about self improvement")
print("A:", get_recommendation("Suggest a book about self improvement"))

Q: I want a mind-bending sci-fi movie
A: "Primer" (2004) is a mind-bending sci-fi masterpiece! This indie film delves into complex time travel concepts, making your head spin. Follow two inventors as they navigate the consequences of altering the timeline. "Primer" requires close attention, but the payoff is pure sci-fi bliss. Prepare to have your mind expanded and challenged!

Q: Suggest a book about self improvement
A: I highly recommend "Atomic Habits" by James Clear. This game-changing book offers practical strategies for building good habits and breaking bad ones. It's packed with actionable advice, real-life examples, and scientific research. Clear's engaging writing style makes self-improvement feel achievable and fun. Get ready to transform your life by forming tiny, yet powerful habits!
